In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants
from ukb_utils import samples
from ko_utils import qc
from ko_utils import analysis

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa018.hpc.in.bmrc.ox.ac.uk:4041
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
input_unphased_path = 'data/unphased/post-qc/ukb_wes_200k_filtered_chr21.mt'

In [4]:
mt2 = hl.read_matrix_table(input_unphased_path)

In [5]:
fam_path = samples.get_fam_path(app_id=11867,wes_200k_only=False,relateds_only=False) # Files with True does not exist
pedigree = hl.Pedigree.read(fam_path)

2021-11-22 18:46:59 Hail: WARN: Found 1 samples with missing sex information (not 1 or 2).
 Missing samples: [{'IID'}]


In [6]:
all_errors, per_fam, per_sample, per_variant = hl.mendel_errors(mt2['GT'], pedigree)

2021-11-22 18:47:05 Hail: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'


In [7]:
input_annotation_path="data/vep/hail/ukb_wes_200k_chr21_vep.ht"
consequence_annotations = hl.read_table(input_annotation_path)
mt2 = mt2.annotate_rows(consequence = consequence_annotations[mt2.row_key]) 

In [8]:
mt2 = mt2.annotate_rows(mendel=per_variant[mt2.locus, mt2.alleles])

In [9]:
by_gene_annotation = analysis.annotation_case_builder(mt2.consequence.vep.worst_csq_for_variant_canonical, use_loftee = True)
mt2 = mt2.annotate_rows(consequence_category = by_gene_annotation)    


In [32]:
mt = mt2

In [44]:
# Annotate with consequence
consequence_annotations = hl.read_table(input_annotation_path)
mt = mt.annotate_rows(consequence = consequence_annotations[mt.row_key]) 
annotation = analysis.annotation_case_builder(mt.consequence.vep.worst_csq_for_variant_canonical, use_loftee = True)
    



In [45]:
mt = mt.annotate_rows(csq_category = annotation) 
mt = mt.annotate_rows(csq_variant = mt.consequence.vep.worst_csq_for_variant_canonical.most_severe_consequence)


In [ ]:

mt = mt.annotate_rows(
        count = hl.struct(
            coding_snp = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1])),
            coding_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]))
        )
) 

In [29]:
# get count of non_ref alleles

mt = mt.annotate_rows(
     count = hl.struct(
        coding_snp = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1])),
        coding_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]))
    )
)

mt = mt.annotate_rows(csq_variant = mt.consequence.vep.worst_csq_for_variant_canonical.most_severe_consequence)
mt = mt.annotate_rows(csq_category = mt.consequence_category)

mt.select_rows('count','mendel', 'csq_variant', 'csq_category').rows().flatten().describe()



----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'locus': locus<GRCh38> 
    'alleles': array<str> 
    'count.coding_snp': int64 
    'count.coding_indel': int64 
    'mendel.errors': int64 
    'csq_variant': str 
    'csq_category': str 
----------------------------------------
Key: []
----------------------------------------


In [29]:
mt.annotate_cols(n_coding_URV_SNP = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_snp(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_coding_URV_indel = hl.agg.count_where(mt.GT.is_non_ref() & hl.is_indel(mt.alleles[0], mt.alleles[1]) & (mt.consequence_category != "non_coding")),
                          n_URV_PTV = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv")),
                          n_URV_PTV_LC = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "ptv_lc")),
                          n_URV_damaging_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "damaging_missense")),
                          n_URV_other_missense = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "other_missense")),
                          n_URV_synonymous = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "synonymous")),
                          n_URV_non_coding = hl.agg.count_where(mt.GT.is_non_ref() & (mt.consequence_category == "non_coding"))
                         )

In [54]:
result_mt = (mt2.group_rows_by(mt2.consequence_category)
                .aggregate(n_non_ref=hl.agg.count_where(mt2.GT.is_non_ref())))

result_mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'consequence_category': str
----------------------------------------
Entry fields:
    'n_non_ref': int64
----------------------------------------
Column key: ['s']
Row key: ['consequence_category']
----------------------------------------


In [51]:
(mt2.group_rows_by(mt2.consequence_category).
            aggregate(
                n_is_non_ref = hl.agg.count_where(mt2.GT.is_non_ref()),
                n_mendel_errors = hl.agg.sum(mt2.mendel.errors),
        
    )
).describe()
 
 

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'consequence_category': str
----------------------------------------
Entry fields:
    'n_is_non_ref': int64
    'n_mendel_errors': int64
----------------------------------------
Column key: ['s']
Row key: ['consequence_category']
----------------------------------------
